*openmobisim tutorials* · [00 · overview](00_getting_started.ipynb) · ← [07 · Validation](07_validation.ipynb) · [09 · Five minimal studies](09_applications.ipynb) → · [documentation](https://openmobisim.github.io)

# 08 · Performance

How long the starter cases take, where the time goes, and how a sample of the demand trades accuracy for speed.
Every run says its time in one line and records its **wall-clock** time by stage (`Run.timings()`, `timings.csv`),
measured on this machine's clock; times differ from run to run and from machine to machine, so they are in no result
and not in the run's fingerprint. The times shown here were measured on a MacBook Pro with an Apple M2 Pro (8 performance and 4 efficiency cores), 32 GB of memory, macOS 26, with nothing else running.

The cities' mornings are congested and large. Amsterdam, Paris and Lyon run **fast** by default: each simulated
traveller stands for five people (ten in Paris); Delft, the smallest, simulates every traveller. Every case shares its
route searches by origin and half hour, so a run takes from ten seconds (Delft) to a minute (Paris) on a
laptop; `case.demand(weight=1)` gives a city's **complete** run, one traveller per person, a few times slower (Delft's
base is one).

**Download** {download}`this notebook <../_static/notebooks/08_performance.ipynb>` or {download}`all ten <../_static/notebooks/openmobisim_notebooks.zip>` (a zip), without their outputs. The Installation section below installs openmobisim; its last command, `openmobisim notebooks`, copies the same ten into a folder and opens the first in JupyterLab.

---

## Installation

> Skip this if `import openmobisim` already works in this notebook. **One install is all there is:**
> `pip install "openmobisim[all]"` brings the simulator and everything the ten tutorials use (JupyterLab,
> matplotlib for the figures, pandas, polars and pyarrow for the tables); nothing is installed later. Find the case
> that matches yours and copy its commands, in order. Each needs Python 3.11 or newer.
>
> **A new conda environment** (any system; openmobisim is on PyPI, so pip installs it into the environment):
>
> ```bash
> conda create -y -n openmobisim python=3.12
> conda activate openmobisim
> pip install "openmobisim[all]"
> openmobisim notebooks
> ```
>
> **A new venv on macOS or Linux** (in a terminal):
>
> ```bash
> python3 -m venv openmobisim
> source openmobisim/bin/activate
> pip install "openmobisim[all]"
> openmobisim notebooks
> ```
>
> **A new venv on Windows** (in the Command Prompt):
>
> ```bat
> py -m venv openmobisim
> openmobisim\Scripts\activate
> pip install "openmobisim[all]"
> openmobisim notebooks
> ```
>
> The last line copies these ten notebooks into `./openmobisim-notebooks` and opens them in JupyterLab. In VS Code or
> PyCharm, open a notebook and choose the `openmobisim` environment's Python as its kernel instead.
>
> **Into the environment this notebook already runs in**: paste this into a new code cell, run it, then restart the
> kernel.
>
> ```python
> %pip install "openmobisim[all]"
> ```

---

In [1]:
import os
import platform
import time

import numpy as np
import openmobisim as ms
import pandas as pd

print(platform.platform(), "·", platform.processor() or platform.machine(), "·", os.cpu_count(), "logical cores")
print(ms.build_info())

macOS-26.6.2-arm64-arm-64bit-Mach-O · arm · 12 logical cores
{'version': '0.1.0', 'git_sha': 'f036acd815fa', 'code_version': 1, 'rng_scheme_version': 1, 'parallel': True, 'target': 'aarch64-apple-darwin', 'profile': 'release'}


## 1 · The starter cases

Each case's default scenario with the package's defaults (a logit model over route sets, the method of successive
averages until converged, the point-queue loading; with modes, every trip choosing among those its class may use).
"read" is reading the network, the timetable and the parkings; "run" is the simulation itself, from the demand to the
written results.

In [2]:
CASES = [("delft", "base", 1), ("nguyendupuis", "base", 1), ("siouxfalls", "base", 1), ("siouxfalls", "congested", 1),
         ("amsterdam", "base", 1), ("paris", "base", 1), ("lyon", "base", 1)]  # fmt: skip


def timed(name, scenario, sample=1):
    """A case's scenario, on one of its travellers in ``sample``, timed."""
    case = ms.examples.case(name)
    t = time.perf_counter()
    sc = case.scenario(scenario)
    if sample > 1:  # the same scenario on a sample of its travellers
        network = case.network()
        transit = case.transit(network)
        sc = ms.Scenario.from_parts(
            network, ms.demand_sample(case.demand(), sample, seed=0), classes=case.classes(),
            transit=transit, parkings=case.parkings() if transit is not None else None,
            **case.info["scenarios"][scenario],
        )  # fmt: skip
    read_s = time.perf_counter() - t
    run = sc.run(f"perf-{name}-{scenario}-{sample}", quiet=True)
    timings = pd.DataFrame(run.timings())
    return run, {
        "travellers simulated": run.completion["total_trips"],
        "people": sum(run.trip_modes()["weight"]),
        "links": run.network.link_count,
        "iterations": len(run.convergence()["gap"]),
        "read (s)": read_s,
        "run (s)": timings.loc[timings["stage"] == "total", "seconds"].iloc[0],
        "mean trip (min)": run.mean_travel_time_s / 60,
    }


runs, rows = {}, {}
for name, scenario, sample in CASES:
    label = f"{name} / {scenario}" + (f", 1 in {sample}" if sample > 1 else "")
    runs[name, scenario, sample], rows[label] = timed(name, scenario, sample)
pd.DataFrame(rows).T.round(2)

/var/folders/q0/vjghqqvj4dvcq_z2rjmvd0jmw3f5y2/T/ipykernel_98515/3079632518.py:19: NoAlternativeWarning: 288 trips (1,440 people, 0.5% of run 'perf-amsterdam-base-1') had no alternative — no mode they may use reached their destination within the limits — and were not simulated. By class (stated mode): 254 'no_vehicle' (choosing), 33 'from_outside_transit' (transit), 1 'car_and_bike' (choosing). Why, by mode offered (trips): car: no_road_route 1; bike: ride_too_long 1; walk: walk_too_long 255; transit: no_stop_in_reach 206, no_transit_journey 82; car_transit: too_short 1; bike_transit: too_short 1. Most between (latitude, longitude, about 1 km): 25 people from (52.39, 4.84) to (52.41, 4.85); 15 people from (52.38, 4.90) to (52.40, 4.87); 15 people from (52.40, 4.91) to (52.41, 4.85); 10 people from (52.35, 4.81) to (52.40, 4.87); 10 people from (52.35, 4.85) to (52.41, 4.82). To change: walk_too_long: mode_options['walk_max_s'] or the class's walk_max_s; ride_too_long: mode_options['bik

,travellers simulated,people,links,iterations,read (s),run (s),mean trip (min)
delft / base,65956.0,65956.0,42699.0,9.0,3.20,10.36,22.94
nguyendupuis / base,2000.0,2000.0,19.0,6.0,0.01,0.04,54.52
siouxfalls / base,180300.0,180300.0,76.0,5.0,1.00,2.46,11.79
siouxfalls / congested,270450.0,270450.0,76.0,6.0,1.49,4.95,21.48
amsterdam / base,57381.0,286905.0,252184.0,6.0,12.84,37.44,24.03
paris / base,84878.0,848780.0,349348.0,5.0,14.90,62.70,20.27
lyon / base,58417.0,292085.0,242580.0,10.0,6.86,51.84,24.99


## 2 · Where the time goes

Amsterdam's run by stage, summed over the iterations: **static routes** (each trip's walk and ride on the layers,
once), **route sets** (the car routes each pair may choose among), the **loadings** (the traffic simulation itself),
the **route updates** (between loadings, new routes on the latest times), the **choices** (routes and itineraries),
the **network gap** (how far from equilibrium), and the results. On this morning the **itinerary choices** take the
largest part (each trip's walk, ride, transit, park-and-ride and bike-and-ride alternatives, searched and weighed:
about half here), then the **route updates** (between loadings, a new best route for every pair on the latest
times: about a quarter); the traffic simulation itself about a sixth, partial loadings included.

In [3]:
t = pd.DataFrame(runs["amsterdam", "base", 1].timings())
t = t[~t["stage"].isin(["total", "network_read"])]
stage = t.groupby("stage", sort=False)["seconds"].sum()
pd.DataFrame({"seconds": stage, "share (%)": 100 * stage / stage.sum()}).round(1)

,seconds,share (%)
stage,,
demand_read,0.0,0.1
setup,0.4,1.0
static_routes,0.9,2.4
itineraries_setup,0.2,0.6
route_sets,0.4,1.1
route_choice,0.4,1.0
itinerary_choice,19.6,52.4
partial_loading,2.1,5.5
route_update,8.6,23.0


## 3 · A sample of the demand

`demand_sample` keeps one traveller in `w`, each standing for `w` times as many people: the run is roughly `w` times
faster. Its vehicles are `w` times larger, so queues are lumpier and the delays a little higher under congestion; a
sample is for screening many scenarios, the fuller demand for the numbers a study reports. Here Amsterdam's fast run
(five people a traveller) against samples of it (ten and twenty people a traveller).

A weight is a small aggregation, not a rescaled network: each simulated traveller is a platoon of as many cars as
people it stands for, each car counted at its full size on every link, on the network's real capacities, storage and
signals (notebook 02). Coarser groups queue in bigger blocks, which is what the table shows.

In [4]:
case = ms.examples.case("amsterdam")
network, transit, parkings = case.network(), None, None
transit = case.transit(network)
parkings = case.parkings()
demand = case.demand()
rows = {}
for w in (4, 2):
    sample = ms.demand_sample(demand, w, seed=0)
    run = ms.Scenario.from_parts(
        network, sample, classes=case.classes(), transit=transit, parkings=parkings,
        **case.info["scenarios"]["base"],
    ).run(f"perf-sample-{w}", quiet=True)  # fmt: skip
    shares = pd.Series(run.trip_modes()["mode"]).value_counts(normalize=True)
    timings = pd.DataFrame(run.timings())
    rows[f"{5 * w} people a traveller"] = {"run (s)": timings.loc[timings["stage"] == "total", "seconds"].iloc[0],
                         "mean trip (min)": run.mean_travel_time_s / 60, "bike (%)": 100 * shares.get("bike", 0),
                         "car (%)": 100 * shares.get("car", 0), "transit (%)": 100 * shares.get("transit", 0)}  # fmt: skip
fast = runs["amsterdam", "base", 1]
shares = pd.Series(fast.trip_modes()["mode"]).value_counts(normalize=True)
rows["5 (the fast run)"] = {"run (s)": pd.DataFrame(fast.timings()).query("stage == 'total'")["seconds"].iloc[0],
                    "mean trip (min)": fast.mean_travel_time_s / 60, "bike (%)": 100 * shares.get("bike", 0),
                    "car (%)": 100 * shares.get("car", 0), "transit (%)": 100 * shares.get("transit", 0)}  # fmt: skip
pd.DataFrame(rows).T.round(2)

/var/folders/q0/vjghqqvj4dvcq_z2rjmvd0jmw3f5y2/T/ipykernel_98515/1177908057.py:12: NoAlternativeWarning: 65 trips (1,300 people, 0.5% of run 'perf-sample-4') had no alternative — no mode they may use reached their destination within the limits — and were not simulated. By class (stated mode): 57 'no_vehicle' (choosing), 7 'from_outside_transit' (transit), 1 'car_and_bike' (choosing). Why, by mode offered (trips): car: no_road_route 1; bike: ride_too_long 1; walk: walk_too_long 58; transit: no_stop_in_reach 52, no_transit_journey 13; car_transit: too_short 1; bike_transit: too_short 1. Most between (latitude, longitude, about 1 km): 60 people from (52.39, 4.84) to (52.41, 4.85); 40 people from (52.37, 4.85) to (52.41, 4.85); 20 people from (52.28, 4.95) to (52.30, 4.95); 20 people from (52.28, 5.00) to (52.30, 4.95); 20 people from (52.29, 5.02) to (52.31, 5.02). To change: walk_too_long: mode_options['walk_max_s'] or the class's walk_max_s; ride_too_long: mode_options['bike_max_s'] or 

/var/folders/q0/vjghqqvj4dvcq_z2rjmvd0jmw3f5y2/T/ipykernel_98515/1177908057.py:12: NoAlternativeWarning: 126 trips (1,260 people, 0.4% of run 'perf-sample-2') had no alternative — no mode they may use reached their destination within the limits — and were not simulated. By class (stated mode): 111 'no_vehicle' (choosing), 14 'from_outside_transit' (transit), 1 'car_and_bike' (choosing). Why, by mode offered (trips): car: no_road_route 1; bike: ride_too_long 1; walk: walk_too_long 112; transit: no_stop_in_reach 97, no_transit_journey 29; car_transit: too_short 1; bike_transit: too_short 1. Most between (latitude, longitude, about 1 km): 40 people from (52.39, 4.84) to (52.41, 4.85); 20 people from (52.36, 4.91) to (52.41, 4.85); 20 people from (52.36, 4.93) to (52.41, 4.84); 20 people from (52.37, 4.85) to (52.41, 4.85); 20 people from (52.38, 4.90) to (52.40, 4.85). To change: walk_too_long: mode_options['walk_max_s'] or the class's walk_max_s; ride_too_long: mode_options['bike_max_s']

,run (s),mean trip (min),bike (%),car (%),transit (%)
20 people a traveller,17.72,26.43,35.15,34.32,28.05
10 people a traveller,23.60,25.90,34.23,35.07,27.98
5 (the fast run),37.44,24.03,33.84,35.56,27.97


**Reading the numbers.** A run's time grows with the travellers simulated, the links, the iterations and the
congestion (a congested morning needs more iterations to settle). The cities' default runs (one simulated traveller
for five people, ten in Paris, every traveller in Delft, on networks of 43 000–350 000 links, with transit, parking
and mode choice) are in the first table; Delft, the smallest city, runs in seconds. A coarser sample is faster still: at ten people a
traveller its mode shares stay within a few tenths of a point, at twenty they move by up to a point and a half, and
its mean trip is a little longer, from its lumpier queues.

**Next:** [09 · Five minimal studies](09_applications.ipynb): five minimal studies to copy and extend: transit frequency, your own utility, accessibility, emissions, a road closed for an hour. The whole series: [00 · overview](00_getting_started.ipynb). The documentation: [openmobisim.github.io](https://openmobisim.github.io). Something wrong or unclear? [Report it](https://github.com/bahmanmdd/openmobisim/issues).